In [ ]:
# ==============================================================================
# File: 05_Launch_Discord_Bot.ipynb (Cell 1 of 2)
# What this notebook does: Mounts Google Drive, clones/syncs https://github.com/Alekhagrawal/ZeroFakev2.git
#   into /content/ZeroFakev2, copies the trained Full ZeroFake checkpoint from Google Drive
#   (/content/drive/MyDrive/ZeroFake_Storage/checkpoints/zerofake/best_model.pt) onto Colab's local
#   SSD (/content/ZeroFakev2/checkpoints/zerofake/best_model.pt), installs all dependencies, and
#   launches BOTH the live Gradio web app (app.py) and the Discord bot (discord_bot.py) together.
# Inputs: /content/drive/MyDrive/ZeroFake_Storage/checkpoints/zerofake/best_model.pt and DISCORD_BOT_TOKEN.
# Outputs: Live public *.gradio.live link + active Discord bot responding to /check and @ZeroFake.
# ==============================================================================

# --- CELL 1: Mount Google Drive, Sync ZeroFakev2, Copy Checkpoint & Install Packages ---
import os
import shutil
import sys
from google.colab import drive

# 1. Safely mount /content/drive
if not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive', ignore_errors=True)
drive.mount('/content/drive')

# 2. Clone or force-sync https://github.com/Alekhagrawal/ZeroFakev2.git into /content/ZeroFakev2
GITHUB_REPO_URL = 'https://github.com/Alekhagrawal/ZeroFakev2.git'
REPO_DIR = '/content/ZeroFakev2'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print(f'[INFO] Fetching and syncing latest commits from {GITHUB_REPO_URL}...')
    !git -C /content/ZeroFakev2 fetch origin
    !git -C /content/ZeroFakev2 reset --hard origin/main
else:
    print(f'[INFO] Cloning {GITHUB_REPO_URL} into {REPO_DIR}...')
    !rm -rf /content/ZeroFakev2
    !git clone https://github.com/Alekhagrawal/ZeroFakev2.git /content/ZeroFakev2

%cd /content/ZeroFakev2
os.environ['PYTHONPATH'] = '/content/ZeroFakev2'
if '/content/ZeroFakev2' not in sys.path:
    sys.path.insert(0, '/content/ZeroFakev2')

# 3. Copy trained checkpoint from Google Drive into /content/ZeroFakev2/checkpoints/zerofake/best_model.pt
STORAGE_DIR = '/content/drive/MyDrive/ZeroFake_Storage'
DRIVE_CKPT_PATH = os.path.join(STORAGE_DIR, 'checkpoints', 'zerofake', 'best_model.pt')
LOCAL_CKPT_DIR = '/content/ZeroFakev2/checkpoints/zerofake'
LOCAL_CKPT_PATH = os.path.join(LOCAL_CKPT_DIR, 'best_model.pt')
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

if not os.path.exists(DRIVE_CKPT_PATH):
    raise FileNotFoundError(
        f'Trained checkpoint not found at {DRIVE_CKPT_PATH}! '
        'Please ensure 02_Train_ZeroFake.ipynb saved best_model.pt to Google Drive.'
    )

shutil.copy2(DRIVE_CKPT_PATH, LOCAL_CKPT_PATH)
ckpt_mb = os.path.getsize(LOCAL_CKPT_PATH) / (1024 * 1024)
print(f'[SUCCESS] Copied trained checkpoint to {LOCAL_CKPT_PATH} ({ckpt_mb:.2f} MB).')

# Sync saved evaluation CSVs and 300-DPI PNG charts if available
DRIVE_RESULTS_DIR = os.path.join(STORAGE_DIR, 'results')
LOCAL_RESULTS_DIR = '/content/ZeroFakev2/results'
if os.path.exists(DRIVE_RESULTS_DIR):
    shutil.copytree(DRIVE_RESULTS_DIR, LOCAL_RESULTS_DIR, dirs_exist_ok=True)
    print(f'[SUCCESS] Synced evaluation tables and plots to {LOCAL_RESULTS_DIR}.')

# 4. Install facenet-pytorch with --no-deps first, then install remaining packages
!pip install -q --no-deps facenet-pytorch
!pip install -q timm opencv-python-headless scikit-learn matplotlib seaborn grad-cam gradio gradio_client discord.py python-dotenv pyyaml tqdm

# 5. Pre-flight CPU dry-run verification on both app.py and discord_bot.py
!PYTHONPATH=/content/ZeroFakev2 python -u app.py --dry-run
!PYTHONPATH=/content/ZeroFakev2 python -u discord_bot.py --dry-run
print('[READY] ZeroFakev2 synced, checkpoint staged on local SSD, and both app.py & discord_bot.py verified!')


In [ ]:
# --- CELL 2: Launch Gradio Server in Background + Start Discord Bot in Foreground ---
# Plain-English explanation:
#   1. Reads DISCORD_BOT_TOKEN from Google Colab Secrets (userdata.get('DISCORD_BOT_TOKEN')),
#      or prompts securely with getpass if the secret has not been added yet.
#   2. Starts `python -u app.py --checkpoint checkpoints/zerofake/best_model.pt` in a background
#      subprocess, streams its startup output to capture any public `https://*.gradio.live` link,
#      and waits until `http://127.0.0.1:7860` responds.
#   3. Sets `ZEROFAKE_URL=http://127.0.0.1:7860` and `DISCORD_BOT_TOKEN`, then runs
#      `python -u discord_bot.py` in the foreground so live Discord bot logs stream directly below.

import getpass
import os
import re
import subprocess
import threading
import time
import urllib.request

%cd /content/ZeroFakev2

# 1. Load DISCORD_BOT_TOKEN from Colab Secrets (with getpass fallback)
token = None
try:
    from google.colab import userdata
    token = userdata.get('DISCORD_BOT_TOKEN')
    if token:
        print('[SUCCESS] Loaded DISCORD_BOT_TOKEN from Google Colab Secrets.')
except Exception:
    token = None

if not token:
    token = os.environ.get('DISCORD_BOT_TOKEN', '').strip()

if not token:
    print('[ACTION REQUIRED] DISCORD_BOT_TOKEN not found in Colab Secrets.')
    token = getpass.getpass('Paste your DISCORD_BOT_TOKEN and press Enter: ').strip()

if not token:
    raise ValueError('DISCORD_BOT_TOKEN cannot be empty! Add it to Colab Secrets or paste it when prompted.')

# 2. Start app.py in a background subprocess and capture its public *.gradio.live share URL
env = os.environ.copy()
env['PYTHONPATH'] = '/content/ZeroFakev2'
env['PYTHONUNBUFFERED'] = '1'

print('[INFO] Launching ZeroFake Gradio server (app.py) in the background on http://127.0.0.1:7860 ...')
gradio_proc = subprocess.Popen(
    [
        'python', '-u', 'app.py',
        '--checkpoint', 'checkpoints/zerofake/best_model.pt',
        '--server_name', '127.0.0.1',
        '--server_port', '7860',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

gradio_share_url = {'url': None}

def _stream_gradio_logs(proc, share_holder):
    pattern = re.compile(r'https://[a-zA-Z0-9.-]+\.gradio\.live')
    for line in iter(proc.stdout.readline, ''):
        clean = line.rstrip()
        if clean:
            print(f'[GRADIO] {clean}', flush=True)
            match = pattern.search(clean)
            if match and not share_holder['url']:
                share_holder['url'] = match.group(0)
                print(f"\n[PUBLIC WEB UI] ZeroFake Live Gradio Dashboard: {share_holder['url']}\n", flush=True)

log_thread = threading.Thread(target=_stream_gradio_logs, args=(gradio_proc, gradio_share_url), daemon=True)
log_thread.start()

# Wait until http://127.0.0.1:7860 responds
LOCAL_GRADIO_URL = 'http://127.0.0.1:7860'
server_ready = False
for attempt in range(90):
    if gradio_proc.poll() is not None:
        raise RuntimeError(f'app.py exited unexpectedly with code {gradio_proc.returncode}')
    try:
        with urllib.request.urlopen(LOCAL_GRADIO_URL, timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        time.sleep(1.0)

if not server_ready:
    raise TimeoutError('Timed out waiting for http://127.0.0.1:7860 to respond.')

# Give Gradio a brief moment to print its public *.gradio.live share URL if running on Colab
for _ in range(8):
    if gradio_share_url['url']:
        break
    time.sleep(1.0)

print(f'[SUCCESS] Gradio server is live at {LOCAL_GRADIO_URL}')
if gradio_share_url['url']:
    print(f"[SUCCESS] Public Gradio Share Link: {gradio_share_url['url']}")

# 3. Configure environment variables and start discord_bot.py in the foreground
os.environ['ZEROFAKE_URL'] = LOCAL_GRADIO_URL
os.environ['DISCORD_BOT_TOKEN'] = token

print('[INFO] Starting ZeroFake Discord Bot in foreground (streaming live logs below)...')
try:
    !PYTHONPATH=/content/ZeroFakev2 python -u discord_bot.py
finally:
    if gradio_proc.poll() is None:
        print('[INFO] Shutting down background Gradio server...')
        gradio_proc.terminate()
